In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import json
import pickle

In [ ]:
# Load Yijun's 5-fold indices
# THIS IS SFOLD NOT KFOLD (variables were historically named)
kfold_pt = 'sfold5_split_train_test_indices.json'
with open(kfold_pt, 'r') as file:
    kfold_indices = json.load(file)

KFOLD INDICES STRUCTURE:
Dict
- first level are keys labeled fold_0 to fold_1
    - in each fold_i value is another dictionary where the keys are 'train' and 'test'
        - the value is a list of indices

In [ ]:
# Load in the geojson with data
pt_data_pt = 'point_pixel_satbands_gt_epsg4326_v3_removednodata.geojson'
point_data_gdf = gpd.read_file(pt_data_pt)
print(point_data_gdf.columns)

# Load in gt with corresponding tax points
json_link_pt = 'mapped-aksdb-point-data-v0.1.json'
with open(json_link_pt, 'r') as file:
    gt_data = json.load(file)

In [ ]:
# Append the tax_column to input data
# Visualize the head
gt_df = pd.DataFrame(gt_data)

# Then merge with your existing dataframe, keeping only necessary columns
point_data_gdf = point_data_gdf.merge(
    gt_df[['id', 'tax_order']], 
    on='id', 
    how='left'  # Use 'left', 'right', 'inner', or 'outer' depending on your needs
)

print(point_data_gdf.head(5))

In [ ]:
#Eliminate all the NA and multi-tax points
print('Shape of raw df: ', point_data_gdf.shape)
point_data_gdf = point_data_gdf[point_data_gdf['tax_order'].notna() & ~point_data_gdf['tax_order'].apply(lambda x: isinstance(x, list))]

# Convert 'tax_order' column to string
point_data_gdf['tax_order'] = point_data_gdf['tax_order'].astype('string')
print('Shape after eliminating NA and multi-tax points: ', point_data_gdf.shape)

In [ ]:
# Eliminate alfisols, aridisols, oxisols
to_remove = ['Alfisols', 'Aridisols', 'Oxisols']
point_data_gdf = point_data_gdf[~point_data_gdf['tax_order'].isin(to_remove)]

print('Shape of gdf post elimination: ', point_data_gdf.shape)

In [ ]:
#Convert 'tax_order' column to an int
point_data_gdf['tax_order'] = point_data_gdf['tax_order'].str.strip()
unique_taxa = sorted(point_data_gdf['tax_order'].unique())
print('Unique Taxa: ', unique_taxa)
tax_order_dict = {name: idx for idx, name in enumerate(unique_taxa)}
idx_to_name_tax_dict = {idx: name for name, idx in tax_order_dict.items()}
point_data_gdf['tax_order_num'] = point_data_gdf['tax_order'].map(tax_order_dict)

In [ ]:
# Define random forest run function
def run_rf(all_ids, kfold_indices, X, y, hyperparameters, save_preds=False, save_rf=False):
    indices = np.array(all_ids)
    
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    class_precisions = {}
    class_recalls = {}
    
    unique_classes = np.unique(y)
    
    for class_label in unique_classes:
        class_precisions[class_label] = []
        class_recalls[class_label] = []
    
    for fold, train_test_dict in kfold_indices.items():
        print('Training Fold: ', fold)

        train_indices = train_test_dict['train']
        test_indices = train_test_dict['test']

        train_rows = np.where(np.isin(indices, train_indices))[0]
        test_rows = np.where(np.isin(indices, test_indices))[0]

        X_train = X[train_rows]
        X_test = X[test_rows]
        y_train = y[train_rows]
        y_test = y[test_rows] 

        print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
        print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

        est = hyperparameters['est']
        verbose = hyperparameters['verbose']
        n_jobs = hyperparameters['n_jobs']
        rf = RandomForestClassifier(n_estimators=est, verbose=verbose, n_jobs=n_jobs)
        rf.fit(X_train, y_train.ravel())
        y_pred = rf.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='macro')
        recall = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')
        precisions_per_class = precision_score(y_test, y_pred, average=None)
        recalls_per_class = recall_score(y_test, y_pred, average=None)
        
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)
        
        unique_y_test = np.unique(y_test)
        for i, class_label in enumerate(np.unique(y_test)):
            class_precision = precisions_per_class[i]
            class_recall = recalls_per_class[i]
            class_precisions[class_label].append(class_precision)
            class_recalls[class_label].append(class_recall)
        
        fold_num = int(fold.split('_')[-1])
        print(f"Fold {fold_num} Accuracy: {accuracy:.4f}")
        print(f"Fold {fold_num} Precision: {precision:.4f}")
        print(f"Fold {fold_num} Recall: {recall:.4f}")
        print(f"Fold {fold_num} F1: {f1:.4f}")
        
        if save_preds:
            test_ids = indices[np.isin(indices, test_indices)]
            preds = {
                "id": test_ids,  
                "gt": y_test.flatten(), 
                "pred": y_pred  
            }
            df_predictions = pd.DataFrame(preds)
            preds_outpath = f"test_predictions_{fold}.csv"
            df_predictions.to_csv(preds_outpath, index=False)

            print(f"Test predictions saved to {preds_outpath}")
            
        if save_rf:
            model_file = f"rf_weights_{fold}.pkl"
            with open(model_file, "wb") as f:
                pickle.dump(rf, f)
            print(f"Model saved to {model_file}")
        
    return (
        fold_accuracies, fold_precisions, fold_recalls, fold_f1s,
        class_precisions, class_recalls
    )

## Experiment Random Forest 1 (RF-1)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B
    - Covariates: Lat, Lon
    - Order: B, G, R, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = point_data_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "RF-1_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 5 (RF-5)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: N/A
    - Covariates: Lat, Lon
    - Order: Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA

In [ ]:
# Loading in data
covariates = ['lon', 'lat']
x_vars = covariates
y_var = ['tax_order_num']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = point_data_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "RF-5_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 2 (RF-2)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34
    - Covariates: Lat, Lon
    - Intuition: Basically all the midsummer channels
    - Order: B, G, R, rededge1, rededge2, rededge3, nir, swir1, 
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA

In [ ]:
# Loading in data 
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = point_data_gdf[x_vars].to_numpy()
y = point_data_gdf[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

#load the order of the indices
indices = point_data_gdf['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-2_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 7 (RF-7)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lat, Lon
    - Covariates: Lat, Lon
    - Order: B, G, R, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
# Loading in covars
covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

In [ ]:
print(point_data_gdf.shape)
print(covar_gdf.shape)

In [ ]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-7_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 8 (RF-8)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lat, Lon
    - Covariates: Lat, Lon
    - Order: B, G, R, ededge1, rededge2, rededge3, nir, swir1, ppt_annual, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
# Loading in covars
covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

In [ ]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-8_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 9 (RF-9)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: R,G,B, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: B, G, R, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
covar_pt = '../data/point_pixel_climate_v1.csv'
covar_gdf = pd.read_csv(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

In [ ]:
merged_df = point_data_gdf.merge(covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'ppt_annual_band_1','tmean_swi_band_1','tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-9_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 10 (RF-10)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: B, G, R, rededge1, rededge2, rededge3, nir, swir1, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
covar_pt = '../data/point_pixel_climate_v1.csv'
covar_gdf = pd.read_csv(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

In [ ]:
merged_df = point_data_gdf.merge(covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 'ppt_annual_band_1','tmean_swi_band_1','tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-10_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 11 (RF-11)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: 25, 26, 27, 28, 29, 30, 31, 33, 34, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january
    - Covariates: Lat, Lon
    - Order: basically satellite channels, env covars like DEM, climate covars
        - B, G, R, rededge1, rededge2, rededge3, nir, swir1, aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, ppt_annual, tmean_swi, tmin_january, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
# Loading in covars
topo_covar_pt = '../data/point_pixel_ifsar_derivs_v1.csv'
topo_covar_gdf = gpd.read_file(topo_covar_pt)
topo_covar_gdf = topo_covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]
topo_covar_gdf['id'] = topo_covar_gdf['id'].astype('int64')

climate_covar_pt = '../data/point_pixel_climate_v1.csv'
climate_covar_gdf = pd.read_csv(climate_covar_pt)
climate_covar_gdf = climate_covar_gdf[['id', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']]

In [ ]:
merged_df = point_data_gdf.merge(topo_covar_gdf, on='id', how='inner')
merged_df = merged_df.merge(climate_covar_gdf, on='id', how='inner')

In [ ]:
# Loading in data
channel_names = ['band_25', 'band_26', 'band_27', 'band_28', 'band_29', 'band_30', 'band_31', 'band_33', 'band_34', 
                    'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 
                     'swi_10_band_1', 'tpi_4_band_1', 'ppt_annual_band_1', 'tmean_swi_band_1', 'tmin_january_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = True, save_rf=True)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-11_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 12 (RF-12)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4
    - Covariates: Lat, Lon
    - Order: env covars like DEM ONLY
        - aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4, Lon, Lat
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
# Loading in covars
covar_pt = 'point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

In [ ]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [ ]:
channel_names = ['aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
covariates = ['lon', 'lat']
x_vars = channel_names + covariates
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-12_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")

## Experiment Random Forest 13 (RF-13)
### Settings:
- Spatial Scale: Pixel Level (i.e. no buffer)
- Inputs: 
    - Channels: aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4
    - Order: env covars like DEM ONLY
        - aspect_4, elevation_full_10m, maxc_4, sl_4, spi, swi_10, tpi_4
- Output: 
    - Soil tax classification, multi-class classification
- Preprocessing: removal of low count classes, removal of NA


In [ ]:
# Loading in covars
covar_pt = 'point_pixel_ifsar_derivs_v1.csv'
covar_gdf = gpd.read_file(covar_pt)
print(covar_gdf.columns)
covar_gdf = covar_gdf[['id', 'aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']]

In [ ]:
point_data_gdf['id'] = point_data_gdf['id'].astype('int64')
covar_gdf['id'] = covar_gdf['id'].astype('int64')
merged_df = pd.merge(point_data_gdf, covar_gdf, on='id', how='inner')

In [ ]:
channel_names = ['aspct_4_band_1', 'elevation_full_10m_3338_band_1', 'maxc_4_band_1', 'sl_4_band_1', 'spi_band_1', 'swi_10_band_1', 'tpi_4_band_1']
x_vars = channel_names 
y_var = ['tax_order_num']

X = merged_df[x_vars].to_numpy()
y = merged_df[y_var].to_numpy()
print('X shape: ', X.shape)
print('Y shape: ', y.shape)

indices = merged_df['id'].to_list()

In [ ]:
#Initialize RF model hyperparameters
hyperparameters = {
    'est': 100,
    'n_jobs': 4,
    'verbose': 1
}

In [ ]:
(
    accuracy, precision, recall, f1,
    class_precisions, class_recalls
) = run_rf(indices, kfold_indices, X, y, hyperparameters, save_preds = False, save_rf=False)

results = {
    "Fold": list(range(0, len(accuracy))),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
}

for class_idx, precision_val_list in class_precisions.items():
    recall_val_list = class_recalls[class_idx]
    p_key = "Precision_" + idx_to_name_tax_dict[class_idx]
    r_key = "Recall_" + idx_to_name_tax_dict[class_idx]
    results[p_key] = precision_val_list
    results[r_key] = recall_val_list

results_df = pd.DataFrame(results)
result_outpath = "results/SFold_taxOrderPred/RF-13_tax-order_results.csv"
results_df.to_csv(result_outpath, index = False)
print(f"Metrics saved to {result_outpath}")